# PS S6E6 - 029 Blend add028 CatBoost v3 check

Experiment: `exp_20260608_029_blend_add028_cat_v3_check`

Purpose:
- Add `028 CatBoost v3 as-is` to the current candidate pool.
- Check whether 028 adds complementary signal to `019 / 027 / 026 / 024 / 020 / 017 / 015 / 018`.
- Evaluate individual CV, pairwise OOF correlation, prediction disagreement, error overlap, and safe blends.
- Save only the best safe blend OOF/pred `.npy` files for later use.

Important:
- `027` is included as the current CV-trusted slot, even though it already contains 020/024/026.
- This notebook is a diagnostic/blend-check notebook. It should not be treated as proof that an in-sample meta model is safe.
- `LogReg / Ridge / ElasticNet` rows are diagnostic only. Final candidates should come from safe methods unless a fold-safe nested stacker is built.
- 028 should not be bias-searched before checking whether it receives non-zero safe blend weight.


In [1]:
# ============================================================
# 0. Imports / Config
# ============================================================

import json
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import spearmanr

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
from sklearn.linear_model import LogisticRegression, RidgeClassifier, Ridge, ElasticNet

warnings.filterwarnings("ignore")

try:
    import yaml
except Exception:
    yaml = None

COMPETITION = "PS S6E6 Predicting Stellar Class"
EXP_ID = "exp_20260608_029_blend_add028_cat_v3_check"
SEED = 42

TRAIN_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/train.csv")
TEST_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/test.csv")
SAMPLE_SUB_PATH = Path("/kaggle/input/competitions/playground-series-s6e6/sample_submission.csv")

# Primary location. The loader below also scans /kaggle/input if a file is not found here.
NPY_DIR = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")
INPUT_ROOT = Path("/kaggle/input")

OUTDIR = Path("/kaggle/working") / EXP_ID
OUTDIR.mkdir(parents=True, exist_ok=True)

ID_COL = "id"
TARGET_COL = "class"

MODEL_SPECS = [
    {
        "key": "015_realmlp_seed42",
        "exp_id": "exp_20260605_015_shared001_realmlp_as_is",
        "family": "RealMLP",
        "role": "stable_single_realmlp_backup",
        "oof": "oof_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "pred": "pred_exp_20260605_015_shared001_realmlp_as_is_proba.npy",
        "cv": 0.9681693449222352,
        "public_lb": 0.96977,
    },
    {
        "key": "017_realmlp_bias",
        "exp_id": "exp_20260606_017_bias_search_on_015_realmlp",
        "family": "RealMLP",
        "role": "previous_best_realmlp_bias_backup",
        "oof": "oof_exp_20260606_017_bias_search_on_015_realmlp_proba.npy",
        "pred": "pred_exp_20260606_017_bias_search_on_015_realmlp_proba.npy",
        "cv": 0.9683022653955233,
        "public_lb": 0.96985,
    },
    {
        "key": "018_xgb_shared003",
        "exp_id": "exp_20260606_018_shared003_xgb_as_is",
        "family": "XGBoost",
        "role": "effective_blend_material",
        "oof": "oof_exp_20260606_018_shared003_xgb_as_is_proba.npy",
        "pred": "pred_exp_20260606_018_shared003_xgb_as_is_proba.npy",
        "cv": 0.965207986418628,
        "public_lb": 0.96578,
    },
    {
        "key": "019_blend_best",
        "exp_id": "exp_20260607_019_blend_add018_xgb_check",
        "family": "Blend",
        "role": "public_confirmed_current_best",
        "oof": "oof_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy",
        "pred": "pred_exp_20260607_019_blend_add018_xgb_check_best_blend_proba.npy",
        "cv": 0.968872315017554,
        "public_lb": 0.97000,
    },
    {
        "key": "020_blend_bias",
        "exp_id": "exp_20260607_020_bias_search_on_019_best_blend",
        "family": "Blend",
        "role": "cv_trusted_attack_reference",
        "oof": "oof_exp_20260607_020_bias_search_on_019_best_blend_proba.npy",
        "pred": "pred_exp_20260607_020_bias_search_on_019_best_blend_proba.npy",
        "cv": 0.9692240443617589,
        "public_lb": 0.96969,
    },
    {
        "key": "024_seed_ensemble_blend",
        "exp_id": "exp_20260608_024_seed_ensemble_and_blend_check",
        "family": "Blend",
        "role": "seed_ensemble_blend_reference",
        "oof": "oof_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_024_seed_ensemble_and_blend_check_best_blend_proba.npy",
        "cv": 0.9694803109983217,
        "public_lb": 0.96958,
    },
    {
        "key": "026_realmlp_v5",
        "exp_id": "exp_20260608_026_realmlp_v5_as_is",
        "family": "RealMLP",
        "role": "realmlp_v5_single_model_candidate",
        "oof": "oof_exp_20260608_026_realmlp_v5_as_is_proba.npy",
        "pred": "pred_exp_20260608_026_realmlp_v5_as_is_proba.npy",
        "cv": 0.9690389777981231,
        "public_lb": 0.96979,
    },
    {
        "key": "027_blend_add026",
        "exp_id": "exp_20260608_027_blend_add026_realmlp_v5_check",
        "family": "Blend",
        "role": "cv_trusted_current_slot",
        "oof": "oof_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy",
        "pred": "pred_exp_20260608_027_blend_add026_realmlp_v5_check_best_blend_proba.npy",
        "cv": 0.9695425457477898,
        "public_lb": 0.96975,
    },
    {
        "key": "028_cat_v3",
        "exp_id": "exp_20260608_028_cat_v3_as_is",
        "family": "CatBoost",
        "role": "catboost_v3_family_aux_material",
        "oof": "oof_exp_20260608_028_cat_v3_as_is_proba.npy",
        "pred": "pred_exp_20260608_028_cat_v3_as_is_proba.npy",
        "cv": 0.9688146470512534,
        "public_lb": 0.96969,
    },
]

# 029 should answer one question first:
# Does 028 receive non-zero useful weight against 019/027 and current components?
BLEND_SETS = {
    # Singles / current slots
    "A_019_only": ["019_blend_best"],
    "B_027_only": ["027_blend_add026"],
    "C_028_only": ["028_cat_v3"],
    "D_026_only": ["026_realmlp_v5"],
    "E_024_only": ["024_seed_ensemble_blend"],
    "F_020_only": ["020_blend_bias"],
    "G_017_only": ["017_realmlp_bias"],
    "H_015_only": ["015_realmlp_seed42"],
    "I_018_only": ["018_xgb_shared003"],

    # Direct add028 checks
    "J_019_028": ["019_blend_best", "028_cat_v3"],
    "K_027_028": ["027_blend_add026", "028_cat_v3"],
    "L_026_028": ["026_realmlp_v5", "028_cat_v3"],
    "M_024_028": ["024_seed_ensemble_blend", "028_cat_v3"],
    "N_020_028": ["020_blend_bias", "028_cat_v3"],
    "O_017_028": ["017_realmlp_bias", "028_cat_v3"],
    "P_015_028": ["015_realmlp_seed42", "028_cat_v3"],
    "Q_018_028": ["018_xgb_shared003", "028_cat_v3"],

    # Current slot vs public slot vs 028
    "R_019_027_028": ["019_blend_best", "027_blend_add026", "028_cat_v3"],
    "S_019_026_028": ["019_blend_best", "026_realmlp_v5", "028_cat_v3"],
    "T_027_026_028": ["027_blend_add026", "026_realmlp_v5", "028_cat_v3"],
    "U_019_024_026_028": ["019_blend_best", "024_seed_ensemble_blend", "026_realmlp_v5", "028_cat_v3"],
    "V_020_024_026_028": ["020_blend_bias", "024_seed_ensemble_blend", "026_realmlp_v5", "028_cat_v3"],
    "W_019_020_024_026_028": ["019_blend_best", "020_blend_bias", "024_seed_ensemble_blend", "026_realmlp_v5", "028_cat_v3"],

    # Older diverse material + 028
    "X_015_017_018_028": ["015_realmlp_seed42", "017_realmlp_bias", "018_xgb_shared003", "028_cat_v3"],
    "Y_017_018_019_028": ["017_realmlp_bias", "018_xgb_shared003", "019_blend_best", "028_cat_v3"],

    # Full diagnostic pool. This contains composite blends and their components, so interpret weights carefully.
    "Z_full_015_017_018_019_020_024_026_027_028": [
        "015_realmlp_seed42", "017_realmlp_bias", "018_xgb_shared003",
        "019_blend_best", "020_blend_bias", "024_seed_ensemble_blend",
        "026_realmlp_v5", "027_blend_add026", "028_cat_v3",
    ],
}

print("EXP_ID:", EXP_ID)
print("OUTDIR:", OUTDIR)
print("NPY_DIR:", NPY_DIR)
print("n_models:", len(MODEL_SPECS))
print("n_blend_sets:", len(BLEND_SETS))


EXP_ID: exp_20260608_029_blend_add028_cat_v3_check
OUTDIR: /kaggle/working/exp_20260608_029_blend_add028_cat_v3_check
NPY_DIR: /kaggle/input/datasets/mizushimatoshihiko/npy-files
n_models: 9
n_blend_sets: 26


In [2]:
# ============================================================
# 1. Utilities
# ============================================================

def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)

def resolve_npy(filename):
    """Resolve an npy path. Prefer NPY_DIR, then scan /kaggle/input.

    This is intentionally more robust than 027 because 028 may be uploaded as a
    separate dataset from older npy files.
    """
    p = NPY_DIR / filename
    if p.exists():
        return p
    matches = list(INPUT_ROOT.rglob(filename)) if INPUT_ROOT.exists() else []
    if len(matches) == 1:
        return matches[0]
    if len(matches) > 1:
        print(f"[WARN] multiple matches for {filename}. Using first:")
        for m in matches[:10]:
            print("  ", m)
        return matches[0]
    raise FileNotFoundError(f"Missing npy file: {filename}. Checked {NPY_DIR} and scanned {INPUT_ROOT}.")

def validate_proba(name, arr, n_rows, n_classes):
    assert arr.shape == (n_rows, n_classes), (name, arr.shape)
    assert np.isfinite(arr).all(), name
    assert np.allclose(arr.sum(axis=1), 1.0, atol=1e-4), name
    assert arr.min() >= -1e-7, (name, arr.min())
    assert arr.max() <= 1.0 + 1e-7, (name, arr.max())

def normalize_rows(p, eps=1e-12):
    p = np.asarray(p, dtype=np.float64)
    p = np.clip(p, eps, None)
    return p / p.sum(axis=1, keepdims=True)

def softmax_np(x, axis=1):
    x = np.asarray(x, dtype=np.float64)
    x = x - np.max(x, axis=axis, keepdims=True)
    ex = np.exp(x)
    return ex / np.sum(ex, axis=axis, keepdims=True)

def proba_to_pred(p):
    return np.argmax(p, axis=1)

def flatten_proba(p):
    return np.asarray(p, dtype=float).reshape(-1)

def corr_pearson(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(np.corrcoef(a, b)[0, 1])

def corr_spearman(a, b):
    a, b = flatten_proba(a), flatten_proba(b)
    if np.std(a) == 0 or np.std(b) == 0:
        return float("nan")
    return float(spearmanr(a, b).correlation)

def class_recalls(y_true, y_pred, class_names):
    out = {}
    for i, cls in enumerate(class_names):
        m = y_true == i
        out[f"recall_{cls}"] = float((y_pred[m] == i).mean()) if m.any() else float("nan")
    return out

def evaluate_proba(y_true, p, class_names):
    pred = proba_to_pred(p)
    res = {"balanced_accuracy": float(balanced_accuracy_score(y_true, pred))}
    res.update(class_recalls(y_true, pred, class_names))
    return res

def rank_normalize_matrix(p):
    out = np.zeros_like(p, dtype=np.float64)
    n = p.shape[0]
    for j in range(p.shape[1]):
        r = pd.Series(p[:, j]).rank(method="average").to_numpy()
        out[:, j] = (r - 1) / max(n - 1, 1)
    return normalize_rows(out)

def logit_transform(p, eps=1e-15):
    p = np.clip(p, eps, 1.0 - eps)
    return np.log(p / (1.0 - p))

def build_meta_features(keys, proba_dict, mode):
    mats = []
    for k in keys:
        p = proba_dict[k]
        if mode == "raw":
            mats.append(p)
        elif mode == "rank":
            mats.append(rank_normalize_matrix(p))
        elif mode == "logit":
            mats.append(logit_transform(p))
        elif mode == "raw_logit":
            mats += [p, logit_transform(p)]
        elif mode == "raw_rank_logit":
            mats += [p, rank_normalize_matrix(p), logit_transform(p)]
        else:
            raise ValueError(mode)
    return np.concatenate(mats, axis=1)

def average_proba(keys, oof_dict, pred_dict=None):
    oof_avg = normalize_rows(np.mean([oof_dict[k] for k in keys], axis=0))
    pred_avg = None
    if pred_dict is not None:
        pred_avg = normalize_rows(np.mean([pred_dict[k] for k in keys], axis=0))
    return oof_avg, pred_avg

def onehot_y(y, n_classes):
    out = np.zeros((len(y), n_classes), dtype=np.float64)
    out[np.arange(len(y)), y] = 1.0
    return out

def hill_climb_weights(y_true, probas, steps=(0.1, 0.05, 0.02, 0.01, 0.005, 0.002), max_iter=300):
    n = len(probas)
    w = np.ones(n, dtype=float) / n
    cur_score = balanced_accuracy_score(y_true, normalize_rows(sum(w[i] * probas[i] for i in range(n))).argmax(axis=1))
    hist = [{"iter": 0, "score": float(cur_score), "weights": w.copy().tolist()}]
    for step in steps:
        improved = True
        it = 0
        while improved and it < max_iter:
            improved = False
            it += 1
            best_score, best_w = cur_score, w.copy()
            for i in range(n):
                for direction in [-1, 1]:
                    cand_w = w.copy()
                    cand_w[i] += direction * step
                    cand_w = np.clip(cand_w, 0.0, None)
                    if cand_w.sum() <= 0:
                        continue
                    cand_w /= cand_w.sum()
                    cand = normalize_rows(sum(cand_w[j] * probas[j] for j in range(n)))
                    score = balanced_accuracy_score(y_true, cand.argmax(axis=1))
                    if score > best_score + 1e-15:
                        best_score, best_w = score, cand_w.copy()
            if best_score > cur_score + 1e-15:
                cur_score, w = best_score, best_w
                improved = True
                hist.append({"iter": len(hist), "step": step, "score": float(cur_score), "weights": w.copy().tolist()})
    return w, float(cur_score), hist

def nm_softmax_weights(y_true, probas, n_restarts=5, maxiter=2500):
    probas = [np.asarray(p, dtype=np.float64) for p in probas]
    n = len(probas)

    def eval_from_theta(theta):
        w = np.exp(theta - np.max(theta))
        w = w / w.sum()
        p = normalize_rows(sum(w[i] * probas[i] for i in range(n)))
        return balanced_accuracy_score(y_true, p.argmax(axis=1)), w

    def objective(theta):
        score, _ = eval_from_theta(theta)
        return -score

    inits = [np.zeros(n)]
    rng = np.random.default_rng(SEED)
    for _ in range(n_restarts - 1):
        inits.append(rng.normal(0, 0.25, size=n))

    best_score, best_w, best_res = -1, None, None
    for init in inits:
        res = minimize(objective, init, method="Nelder-Mead", options={"maxiter": maxiter, "xatol": 1e-8, "fatol": 1e-12})
        score, w = eval_from_theta(res.x)
        if score > best_score:
            best_score, best_w, best_res = score, w, res
    return best_w, float(best_score), best_res

def disagreement_and_error_overlap(y_true, p1, p2):
    pred1 = p1.argmax(axis=1)
    pred2 = p2.argmax(axis=1)
    wrong1 = pred1 != y_true
    wrong2 = pred2 != y_true
    both_wrong = wrong1 & wrong2
    either_wrong = wrong1 | wrong2
    return {
        "disagreement_rate": float((pred1 != pred2).mean()),
        "wrong_rate_left": float(wrong1.mean()),
        "wrong_rate_right": float(wrong2.mean()),
        "both_wrong_rate": float(both_wrong.mean()),
        "error_overlap_jaccard": float(both_wrong.sum() / max(either_wrong.sum(), 1)),
        "left_wrong_right_correct_rate": float((wrong1 & ~wrong2).mean()),
        "right_wrong_left_correct_rate": float((wrong2 & ~wrong1).mean()),
    }


In [3]:
# ============================================================
# 2. Load data and OOF/pred
# ============================================================

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    if not p.exists():
        raise FileNotFoundError(p)

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample = pd.read_csv(SAMPLE_SUB_PATH)

le = LabelEncoder()
y = le.fit_transform(train[TARGET_COL].astype(str))
class_names = le.classes_.tolist()
n_classes = len(class_names)
assert class_names == ["GALAXY", "QSO", "STAR"], class_names

oof, pred, resolved_specs = {}, {}, []
for spec in MODEL_SPECS:
    key = spec["key"]
    oof_path = resolve_npy(spec["oof"])
    pred_path = resolve_npy(spec["pred"])
    oof_arr = np.load(oof_path)
    pred_arr = np.load(pred_path)
    validate_proba(f"oof:{key}", oof_arr, len(train), n_classes)
    validate_proba(f"pred:{key}", pred_arr, len(test), n_classes)
    oof[key] = oof_arr.astype(np.float64)
    pred[key] = pred_arr.astype(np.float64)
    s = spec.copy()
    s["resolved_oof_path"] = str(oof_path)
    s["resolved_pred_path"] = str(pred_path)
    resolved_specs.append(s)
    print(f"loaded {key}: {oof_arr.shape} / {pred_arr.shape}")

model_keys = [s["key"] for s in resolved_specs]
print("class_names:", class_names)
print("loaded keys:", model_keys)


loaded 015_realmlp_seed42: (577347, 3) / (247435, 3)
loaded 017_realmlp_bias: (577347, 3) / (247435, 3)
loaded 018_xgb_shared003: (577347, 3) / (247435, 3)
loaded 019_blend_best: (577347, 3) / (247435, 3)
loaded 020_blend_bias: (577347, 3) / (247435, 3)
loaded 024_seed_ensemble_blend: (577347, 3) / (247435, 3)
loaded 026_realmlp_v5: (577347, 3) / (247435, 3)
loaded 027_blend_add026: (577347, 3) / (247435, 3)
loaded 028_cat_v3: (577347, 3) / (247435, 3)
class_names: ['GALAXY', 'QSO', 'STAR']
loaded keys: ['015_realmlp_seed42', '017_realmlp_bias', '018_xgb_shared003', '019_blend_best', '020_blend_bias', '024_seed_ensemble_blend', '026_realmlp_v5', '027_blend_add026', '028_cat_v3']


In [4]:
# ============================================================
# 3. Individual scores and pairwise diagnostics
# ============================================================

rows = []
for spec in resolved_specs:
    key = spec["key"]
    p = oof[key]
    y_pred = proba_to_pred(p)
    score = balanced_accuracy_score(y, y_pred)
    row = {
        "key": key,
        "exp_id": spec["exp_id"],
        "family": spec["family"],
        "role": spec["role"],
        "cv_from_memo": spec.get("cv", np.nan),
        "public_lb": spec.get("public_lb", np.nan),
        "recomputed_oof_cv": float(score),
        "delta_recomputed_minus_memo": float(score - spec.get("cv", score)),
    }
    row.update(class_recalls(y, y_pred, class_names))
    rows.append(row)

individual_df = pd.DataFrame(rows).sort_values("recomputed_oof_cv", ascending=False).reset_index(drop=True)
display(individual_df)
individual_df.to_csv(OUTDIR / "individual_scores.csv", index=False)

pair_rows = []
for a, b in combinations(model_keys, 2):
    diag = disagreement_and_error_overlap(y, oof[a], oof[b])
    pair_rows.append({
        "left": a,
        "right": b,
        "pearson_flat_proba": corr_pearson(oof[a], oof[b]),
        "spearman_flat_proba": corr_spearman(oof[a], oof[b]),
        **diag,
    })

pairwise_df = pd.DataFrame(pair_rows)
pairwise_df = pairwise_df.sort_values(["pearson_flat_proba", "error_overlap_jaccard"], ascending=[True, True]).reset_index(drop=True)
display(pairwise_df)
pairwise_df.to_csv(OUTDIR / "pairwise_oof_correlation.csv", index=False)

# Focus table for 028.
focus_028_df = pairwise_df[(pairwise_df["left"] == "028_cat_v3") | (pairwise_df["right"] == "028_cat_v3")].copy()
focus_028_df = focus_028_df.sort_values("pearson_flat_proba").reset_index(drop=True)
display(focus_028_df)
focus_028_df.to_csv(OUTDIR / "pairwise_oof_correlation_focus_028.csv", index=False)


,key,exp_id,family,role,cv_from_memo,public_lb,recomputed_oof_cv,delta_recomputed_minus_memo,recall_GALAXY,recall_QSO,recall_STAR
0,027_blend_add026,exp_20260608_027_blend_add026_realmlp_v5_check,Blend,cv_trusted_current_slot,0.969543,0.96975,0.969543,0.0,0.961479,0.976439,0.970710
1,024_seed_ensemble_blend,exp_20260608_024_seed_ensemble_and_blend_check,Blend,seed_ensemble_blend_reference,0.969480,0.96958,0.969480,0.0,0.961956,0.976174,0.970311
2,020_blend_bias,exp_20260607_020_bias_search_on_019_best_blend,Blend,cv_trusted_attack_reference,0.969224,0.96969,0.969224,0.0,0.961137,0.976200,0.970335
3,026_realmlp_v5,exp_20260608_026_realmlp_v5_as_is,RealMLP,realmlp_v5_single_model_candidate,0.969039,0.96979,0.969039,0.0,0.959505,0.976431,0.971181
4,019_blend_best,exp_20260607_019_blend_add018_xgb_check,Blend,public_confirmed_current_best,0.968872,0.97000,0.968872,0.0,0.965479,0.974441,0.966696
5,028_cat_v3,exp_20260608_028_cat_v3_as_is,CatBoost,catboost_v3_family_aux_material,0.968815,0.96969,0.968815,0.0,0.960099,0.974826,0.971520
6,017_realmlp_bias,exp_20260606_017_bias_search_on_015_realmlp,RealMLP,previous_best_realmlp_bias_backup,0.968302,0.96985,0.968302,0.0,0.959889,0.975867,0.969150
7,015_realmlp_seed42,exp_20260605_015_shared001_realmlp_as_is,RealMLP,stable_single_realmlp_backup,0.968169,0.96977,0.968169,0.0,0.962234,0.976098,0.966177
8,018_xgb_shared003,exp_20260606_018_shared003_xgb_as_is,XGBoost,effective_blend_material,0.965208,0.96578,0.965208,0.0,0.966870,0.972043,0.956711


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,017_realmlp_bias,018_xgb_shared003,0.990670,0.908650,0.018476,0.035542,0.033536,0.025439,0.582933,0.010103,0.008097
1,015_realmlp_seed42,018_xgb_shared003,0.991146,0.918772,0.017757,0.034388,0.033536,0.025227,0.590848,0.009161,0.008309
2,018_xgb_shared003,026_realmlp_v5,0.991382,0.794941,0.017291,0.033536,0.035388,0.025946,0.603716,0.007590,0.009441
3,018_xgb_shared003,028_cat_v3,0.992356,0.969802,0.016441,0.033536,0.035277,0.026291,0.618289,0.007245,0.008986
4,015_realmlp_seed42,028_cat_v3,0.993619,0.913554,0.015279,0.034388,0.035277,0.027358,0.646647,0.007030,0.007919
5,017_realmlp_bias,028_cat_v3,0.993673,0.904245,0.015462,0.035542,0.035277,0.027832,0.647474,0.007709,0.007444
6,018_xgb_shared003,027_blend_add026,0.994234,0.866554,0.014541,0.033536,0.034163,0.026686,0.650661,0.006850,0.007477
7,018_xgb_shared003,024_seed_ensemble_blend,0.994640,0.938731,0.014059,0.033536,0.033962,0.026821,0.659357,0.006715,0.007141
8,026_realmlp_v5,028_cat_v3,0.994791,0.804927,0.013787,0.035388,0.035277,0.028589,0.679483,0.006798,0.006687
9,019_blend_best,028_cat_v3,0.995654,0.926925,0.012703,0.032528,0.035277,0.027673,0.689555,0.004855,0.007604


,left,right,pearson_flat_proba,spearman_flat_proba,disagreement_rate,wrong_rate_left,wrong_rate_right,both_wrong_rate,error_overlap_jaccard,left_wrong_right_correct_rate,right_wrong_left_correct_rate
0,018_xgb_shared003,028_cat_v3,0.992356,0.969802,0.016441,0.033536,0.035277,0.026291,0.618289,0.007245,0.008986
1,015_realmlp_seed42,028_cat_v3,0.993619,0.913554,0.015279,0.034388,0.035277,0.027358,0.646647,0.007030,0.007919
2,017_realmlp_bias,028_cat_v3,0.993673,0.904245,0.015462,0.035542,0.035277,0.027832,0.647474,0.007709,0.007444
3,026_realmlp_v5,028_cat_v3,0.994791,0.804927,0.013787,0.035388,0.035277,0.028589,0.679483,0.006798,0.006687
4,019_blend_best,028_cat_v3,0.995654,0.926925,0.012703,0.032528,0.035277,0.027673,0.689555,0.004855,0.007604
5,024_seed_ensemble_blend,028_cat_v3,0.995894,0.936706,0.012668,0.033962,0.035277,0.028413,0.695940,0.005550,0.006864
6,020_blend_bias,028_cat_v3,0.995895,0.925055,0.012687,0.034489,0.035277,0.028659,0.697173,0.005830,0.006618
7,027_blend_add026,028_cat_v3,0.995986,0.871535,0.012578,0.034163,0.035277,0.028563,0.698771,0.005600,0.006713


In [5]:
# ============================================================
# 4. Blend diagnostics
# ============================================================

blend_rows = {}
blend_probas = {}

def record_blend(set_name, method, keys, oof_p, test_p=None, weights=None, extra=None):
    ev = evaluate_proba(y, oof_p, class_names)
    row = {
        "set_name": set_name,
        "method": method,
        "keys": ",".join(keys),
        "n_models": len(keys),
        "balanced_accuracy": ev["balanced_accuracy"],
        "includes_028": "028_cat_v3" in keys,
        "weight_028": None,
        "weights_json": json.dumps({k: float(w) for k, w in zip(keys, weights)}, ensure_ascii=False) if weights is not None else None,
    }
    if weights is not None and "028_cat_v3" in keys:
        row["weight_028"] = float(dict(zip(keys, weights)).get("028_cat_v3", 0.0))
    row.update({k: v for k, v in ev.items() if k != "balanced_accuracy"})
    if extra:
        row.update(extra)
    blend_rows[(set_name, method)] = row
    if test_p is not None:
        blend_probas[(set_name, method)] = (normalize_rows(oof_p), normalize_rows(test_p))

for set_name, keys in BLEND_SETS.items():
    print("\n===", set_name, keys, "===")

    # 1) Simple average.
    avg_oof, avg_pred = average_proba(keys, oof, pred)
    record_blend(set_name, "avg", keys, avg_oof, avg_pred)

    # 2) Hill climb nonnegative raw probabilities.
    try:
        w, score, hist = hill_climb_weights(y, [oof[k] for k in keys])
        hc_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        hc_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "hc_nonnegative_raw", keys, hc_oof, hc_pred, weights=w, extra={"hc_score_internal": score, "hc_n_steps": len(hist)})
    except Exception as e:
        print(f"[WARN] HC failed: {set_name}: {e}")

    # 3) Nelder-Mead softmax weights.
    try:
        w, score, res = nm_softmax_weights(y, [oof[k] for k in keys])
        nm_oof = normalize_rows(sum(w[i] * oof[keys[i]] for i in range(len(keys))))
        nm_pred = normalize_rows(sum(w[i] * pred[keys[i]] for i in range(len(keys))))
        record_blend(set_name, "nm_softmax_raw", keys, nm_oof, nm_pred, weights=w, extra={"nm_score_internal": score, "nm_success": bool(res.success), "nm_message": str(res.message)})
    except Exception as e:
        print(f"[WARN] NM failed: {set_name}: {e}")

    # 4) In-sample meta models. Diagnostic only.
    for mode in ["raw", "rank", "logit", "raw_logit", "raw_rank_logit"]:
        X_meta = build_meta_features(keys, oof, mode)
        X_test_meta = build_meta_features(keys, pred, mode)
        try:
            lr = LogisticRegression(C=0.05, multi_class="multinomial", solver="lbfgs", max_iter=2000, random_state=SEED)
            lr.fit(X_meta, y)
            record_blend(set_name, f"logreg_{mode}", keys, lr.predict_proba(X_meta), lr.predict_proba(X_test_meta), extra={"C": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] logreg failed: {set_name} {mode}: {e}")
        try:
            rc = RidgeClassifier(alpha=10.0, random_state=SEED)
            rc.fit(X_meta, y)
            record_blend(set_name, f"ridgeclf_{mode}", keys, softmax_np(rc.decision_function(X_meta)), softmax_np(rc.decision_function(X_test_meta)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridgeclf failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            rr = Ridge(alpha=10.0, random_state=SEED)
            rr.fit(X_meta, y_oh)
            record_blend(set_name, f"ridge_reg_{mode}", keys, normalize_rows(np.clip(rr.predict(X_meta), 1e-12, None)), normalize_rows(np.clip(rr.predict(X_test_meta), 1e-12, None)), extra={"alpha": 10.0, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] ridge_reg failed: {set_name} {mode}: {e}")
        try:
            y_oh = onehot_y(y, n_classes)
            oof_cols, test_cols = [], []
            for c in range(n_classes):
                en = ElasticNet(alpha=0.0005, l1_ratio=0.05, random_state=SEED, max_iter=2000)
                en.fit(X_meta, y_oh[:, c])
                oof_cols.append(en.predict(X_meta))
                test_cols.append(en.predict(X_test_meta))
            en_oof = normalize_rows(np.clip(np.vstack(oof_cols).T, 1e-12, None))
            en_pred = normalize_rows(np.clip(np.vstack(test_cols).T, 1e-12, None))
            record_blend(set_name, f"elasticnet_reg_{mode}", keys, en_oof, en_pred, extra={"alpha": 0.0005, "l1_ratio": 0.05, "risk": "in_sample_meta_screening"})
        except Exception as e:
            print(f"[WARN] elasticnet_reg failed: {set_name} {mode}: {e}")

blend_df = pd.DataFrame(list(blend_rows.values())).sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(blend_df.head(200))
blend_df.to_csv(OUTDIR / "blend_diagnostics.csv", index=False)

safe_methods = ["avg", "hc_nonnegative_raw", "nm_softmax_raw"]
safe_blend_df = blend_df[blend_df["method"].isin(safe_methods)].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(safe_blend_df.head(80))
safe_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics.csv", index=False)

focus_028_blend_df = safe_blend_df[safe_blend_df["includes_028"]].copy().sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
display(focus_028_blend_df.head(80))
focus_028_blend_df.to_csv(OUTDIR / "safe_blend_diagnostics_focus_028.csv", index=False)



=== A_019_only ['019_blend_best'] ===

=== B_027_only ['027_blend_add026'] ===

=== C_028_only ['028_cat_v3'] ===

=== D_026_only ['026_realmlp_v5'] ===

=== E_024_only ['024_seed_ensemble_blend'] ===

=== F_020_only ['020_blend_bias'] ===

=== G_017_only ['017_realmlp_bias'] ===

=== H_015_only ['015_realmlp_seed42'] ===

=== I_018_only ['018_xgb_shared003'] ===

=== J_019_028 ['019_blend_best', '028_cat_v3'] ===

=== K_027_028 ['027_blend_add026', '028_cat_v3'] ===

=== L_026_028 ['026_realmlp_v5', '028_cat_v3'] ===

=== M_024_028 ['024_seed_ensemble_blend', '028_cat_v3'] ===

=== N_020_028 ['020_blend_bias', '028_cat_v3'] ===

=== O_017_028 ['017_realmlp_bias', '028_cat_v3'] ===

=== P_015_028 ['015_realmlp_seed42', '028_cat_v3'] ===

=== Q_018_028 ['018_xgb_shared003', '028_cat_v3'] ===

=== R_019_027_028 ['019_blend_best', '027_blend_add026', '028_cat_v3'] ===

=== S_019_026_028 ['019_blend_best', '026_realmlp_v5', '028_cat_v3'] ===

=== T_027_026_028 ['027_blend_add026', '026_re

,set_name,method,keys,n_models,balanced_accuracy,includes_028,weight_028,weights_json,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,T_027_026_028,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3",3,0.970002,True,0.446846,"{""027_blend_add026"": 0.28409019375958683, ""026...",0.961315,0.975867,0.972825,0.970002,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T_027_026_028,nm_softmax_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3",3,0.969977,True,0.406822,"{""027_blend_add026"": 0.3395752468613964, ""026_...",0.961431,0.975893,0.972608,NaN,NaN,0.969977,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
2,R_019_027_028,hc_nonnegative_raw,"019_blend_best,027_blend_add026,028_cat_v3",3,0.969972,True,0.388765,"{""019_blend_best"": 0.10440266632584404, ""027_b...",0.962284,0.975654,0.971979,0.969972,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,V_020_024_026_028,hc_nonnegative_raw,"020_blend_bias,024_seed_ensemble_blend,026_rea...",4,0.969966,True,0.350294,"{""020_blend_bias"": 0.3070045562504073, ""024_se...",0.961566,0.975952,0.972378,0.969966,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,S_019_026_028,hc_nonnegative_raw,"019_blend_best,026_realmlp_v5,028_cat_v3",3,0.969951,True,0.399918,"{""019_blend_best"": 0.2924524674182052, ""026_re...",0.962512,0.975483,0.971858,0.969951,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,B_027_only,elasticnet_reg_raw_rank_logit,027_blend_add026,1,0.968141,False,NaN,None,0.969450,0.973451,0.961523,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,0.0005,0.05
196,C_028_only,ridge_reg_raw_rank_logit,028_cat_v3,1,0.968140,True,NaN,None,0.965426,0.973784,0.965210,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN
197,C_028_only,ridgeclf_raw_rank_logit,028_cat_v3,1,0.968140,True,NaN,None,0.965426,0.973784,0.965210,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,10.0000,NaN
198,G_017_only,elasticnet_reg_raw,017_realmlp_bias,1,0.968110,False,NaN,None,0.962515,0.975338,0.966479,NaN,NaN,NaN,NaN,NaN,NaN,in_sample_meta_screening,0.0005,0.05


,set_name,method,keys,n_models,balanced_accuracy,includes_028,weight_028,weights_json,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,T_027_026_028,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3",3,0.970002,True,0.446846,"{""027_blend_add026"": 0.28409019375958683, ""026...",0.961315,0.975867,0.972825,0.970002,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T_027_026_028,nm_softmax_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3",3,0.969977,True,0.406822,"{""027_blend_add026"": 0.3395752468613964, ""026_...",0.961431,0.975893,0.972608,NaN,NaN,0.969977,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
2,R_019_027_028,hc_nonnegative_raw,"019_blend_best,027_blend_add026,028_cat_v3",3,0.969972,True,0.388765,"{""019_blend_best"": 0.10440266632584404, ""027_b...",0.962284,0.975654,0.971979,0.969972,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,V_020_024_026_028,hc_nonnegative_raw,"020_blend_bias,024_seed_ensemble_blend,026_rea...",4,0.969966,True,0.350294,"{""020_blend_bias"": 0.3070045562504073, ""024_se...",0.961566,0.975952,0.972378,0.969966,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,S_019_026_028,hc_nonnegative_raw,"019_blend_best,026_realmlp_v5,028_cat_v3",3,0.969951,True,0.399918,"{""019_blend_best"": 0.2924524674182052, ""026_re...",0.962512,0.975483,0.971858,0.969951,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73,H_015_only,avg,015_realmlp_seed42,1,0.968169,False,NaN,None,0.962234,0.976098,0.966177,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
74,H_015_only,hc_nonnegative_raw,015_realmlp_seed42,1,0.968169,False,NaN,"{""015_realmlp_seed42"": 1.0}",0.962234,0.976098,0.966177,0.968169,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75,I_018_only,avg,018_xgb_shared003,1,0.965208,False,NaN,None,0.966870,0.972043,0.956711,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
76,I_018_only,nm_softmax_raw,018_xgb_shared003,1,0.965208,False,NaN,"{""018_xgb_shared003"": 1.0}",0.966870,0.972043,0.956711,NaN,NaN,0.965208,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN


,set_name,method,keys,n_models,balanced_accuracy,includes_028,weight_028,weights_json,recall_GALAXY,recall_QSO,recall_STAR,hc_score_internal,hc_n_steps,nm_score_internal,nm_success,nm_message,C,risk,alpha,l1_ratio
0,T_027_026_028,hc_nonnegative_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3",3,0.970002,True,0.446846,"{""027_blend_add026"": 0.28409019375958683, ""026...",0.961315,0.975867,0.972825,0.970002,6.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,T_027_026_028,nm_softmax_raw,"027_blend_add026,026_realmlp_v5,028_cat_v3",3,0.969977,True,0.406822,"{""027_blend_add026"": 0.3395752468613964, ""026_...",0.961431,0.975893,0.972608,NaN,NaN,0.969977,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
2,R_019_027_028,hc_nonnegative_raw,"019_blend_best,027_blend_add026,028_cat_v3",3,0.969972,True,0.388765,"{""019_blend_best"": 0.10440266632584404, ""027_b...",0.962284,0.975654,0.971979,0.969972,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,V_020_024_026_028,hc_nonnegative_raw,"020_blend_bias,024_seed_ensemble_blend,026_rea...",4,0.969966,True,0.350294,"{""020_blend_bias"": 0.3070045562504073, ""024_se...",0.961566,0.975952,0.972378,0.969966,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,S_019_026_028,hc_nonnegative_raw,"019_blend_best,026_realmlp_v5,028_cat_v3",3,0.969951,True,0.399918,"{""019_blend_best"": 0.2924524674182052, ""026_re...",0.962512,0.975483,0.971858,0.969951,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,S_019_026_028,nm_softmax_raw,"019_blend_best,026_realmlp_v5,028_cat_v3",3,0.969948,True,0.405236,"{""019_blend_best"": 0.2516282889873638, ""026_re...",0.962295,0.975534,0.972015,NaN,NaN,0.969948,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
6,U_019_024_026_028,hc_nonnegative_raw,"019_blend_best,024_seed_ensemble_blend,026_rea...",4,0.969947,True,0.395852,"{""019_blend_best"": 0.19636458719391936, ""024_s...",0.962467,0.975551,0.971822,0.969947,9.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,K_027_028,hc_nonnegative_raw,"027_blend_add026,028_cat_v3",2,0.969947,True,0.434589,"{""027_blend_add026"": 0.5654113497250752, ""028_...",0.961781,0.975705,0.972354,0.969947,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,M_024_028,nm_softmax_raw,"024_seed_ensemble_blend,028_cat_v3",2,0.969935,True,0.387336,"{""024_seed_ensemble_blend"": 0.612664294403852,...",0.962130,0.975731,0.971943,NaN,NaN,0.969935,True,Optimization terminated successfully.,NaN,NaN,NaN,NaN
9,M_024_028,hc_nonnegative_raw,"024_seed_ensemble_blend,028_cat_v3",2,0.969934,True,0.387009,"{""024_seed_ensemble_blend"": 0.6129909499533569...",0.962128,0.975731,0.971943,0.969934,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# ============================================================
# 5. Save best safe blend
# ============================================================

safe_blend_df = safe_blend_df.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
best_row = safe_blend_df.iloc[0].to_dict()
best_set_name = best_row["set_name"]
best_method = best_row["method"]
best_keys = best_row["keys"].split(",")

print("Best safe blend:")
print(json.dumps(best_row, indent=2, ensure_ascii=False))

if best_method == "avg":
    best_oof_proba, best_pred_proba = average_proba(best_keys, oof, pred)
else:
    weights_json = json.loads(best_row["weights_json"])
    w = np.array([weights_json[k] for k in best_keys], dtype=float)
    best_oof_proba = normalize_rows(sum(w[i] * oof[best_keys[i]] for i in range(len(best_keys))))
    best_pred_proba = normalize_rows(sum(w[i] * pred[best_keys[i]] for i in range(len(best_keys))))

validate_proba("best_oof_proba", best_oof_proba, len(train), n_classes)
validate_proba("best_pred_proba", best_pred_proba, len(test), n_classes)

best_oof_score = balanced_accuracy_score(y, best_oof_proba.argmax(axis=1))
print("best_oof_score:", best_oof_score)

best_oof_npy = OUTDIR / "oof_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy"
best_pred_npy = OUTDIR / "pred_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy"
np.save(best_oof_npy, best_oof_proba.astype(np.float32))
np.save(best_pred_npy, best_pred_proba.astype(np.float32))

best_labels = le.inverse_transform(best_pred_proba.argmax(axis=1))
submission = pd.DataFrame({ID_COL: test[ID_COL].values, TARGET_COL: best_labels})
assert submission.shape[0] == sample.shape[0]
assert submission.columns.tolist() == sample.columns.tolist()
assert submission[ID_COL].equals(sample[ID_COL])

best_submission_path = OUTDIR / "submission_exp_20260608_029_blend_add028_cat_v3_check_best_blend.csv"
submission.to_csv(best_submission_path, index=False)

best_info = {
    "exp_id": EXP_ID,
    "best_set_name": best_set_name,
    "best_method": best_method,
    "best_keys": best_keys,
    "cv_score": float(best_oof_score),
    "includes_028": "028_cat_v3" in best_keys,
    "weight_028": best_row.get("weight_028"),
    "weights_json": best_row.get("weights_json"),
    "oof_path": str(best_oof_npy),
    "pred_path": str(best_pred_npy),
    "submission_path": str(best_submission_path),
    "row": best_row,
    "note": "Best safe blend excludes in-sample meta screening methods.",
}
save_json(best_info, OUTDIR / "best_blend_info.json")

saved_submission_df = pd.DataFrame([{
    "set_name": best_set_name,
    "method": best_method,
    "balanced_accuracy": float(best_oof_score),
    "includes_028": "028_cat_v3" in best_keys,
    "weight_028": best_row.get("weight_028"),
    "submission": best_submission_path.name,
    "oof_proba": best_oof_npy.name,
    "pred_proba": best_pred_npy.name,
}])
display(saved_submission_df)
saved_submission_df.to_csv(OUTDIR / "saved_submission_candidates.csv", index=False)


Best safe blend:
{
  "set_name": "T_027_026_028",
  "method": "hc_nonnegative_raw",
  "keys": "027_blend_add026,026_realmlp_v5,028_cat_v3",
  "n_models": 3,
  "balanced_accuracy": 0.9700023026377228,
  "includes_028": true,
  "weight_028": 0.4468464883027371,
  "weights_json": "{\"027_blend_add026\": 0.28409019375958683, \"026_realmlp_v5\": 0.26906331793767624, \"028_cat_v3\": 0.4468464883027371}",
  "recall_GALAXY": 0.9613145067288333,
  "recall_QSO": 0.9758671026010944,
  "recall_STAR": 0.9728252985832406,
  "hc_score_internal": 0.9700023026377228,
  "hc_n_steps": 6.0,
  "nm_score_internal": NaN,
  "nm_success": NaN,
  "nm_message": NaN,
  "C": NaN,
  "risk": NaN,
  "alpha": NaN,
  "l1_ratio": NaN
}
best_oof_score: 0.9700023026377228


,set_name,method,balanced_accuracy,includes_028,weight_028,submission,oof_proba,pred_proba
0,T_027_026_028,hc_nonnegative_raw,0.970002,True,0.446846,submission_exp_20260608_029_blend_add028_cat_v...,oof_exp_20260608_029_blend_add028_cat_v3_check...,pred_exp_20260608_029_blend_add028_cat_v3_chec...


In [7]:
# ============================================================
# 6. Save summary / memo
# ============================================================

reference_scores = {
    "019_cv": 0.968872315017554,
    "019_public_lb": 0.97000,
    "027_cv": 0.9695425457477898,
    "027_public_lb": 0.96975,
    "026_cv": 0.9690389777981231,
    "026_public_lb": 0.96979,
    "024_cv": 0.9694803109983217,
    "024_public_lb": 0.96958,
    "020_cv": 0.9692240443617589,
    "020_public_lb": 0.96969,
    "017_cv": 0.9683022653955233,
    "017_public_lb": 0.96985,
    "015_cv": 0.9681693449222352,
    "015_public_lb": 0.96977,
    "018_cv": 0.965207986418628,
    "018_public_lb": 0.96578,
    "028_cv": 0.9688146470512534,
    "028_public_lb": 0.96969,
}

# Important derived checks for judging 028.
best_028_safe = focus_028_blend_df.iloc[0].to_dict() if len(focus_028_blend_df) else None
best_safe_weight_028 = best_info.get("weight_028")

judgement = {
    "reference_scores": reference_scores,
    "individual_best": {
        "key": individual_df.iloc[0]["key"],
        "cv": float(individual_df.iloc[0]["recomputed_oof_cv"]),
        "public_lb": float(individual_df.iloc[0]["public_lb"]),
    },
    "best_safe_blend": best_info,
    "best_safe_blend_with_028": best_028_safe,
    "main_questions": {
        "does_028_add_to_019": "Check J/R/S/U/W/Y/Z safe methods and 028 weight.",
        "does_028_add_to_027": "Check K/R/T/Z safe methods and 028 weight.",
        "does_028_add_to_core_components": "Check V/W/Z safe methods, especially 028 weight against 020/024/026.",
        "should_bias_search_028_next": "No. Only consider if 028 receives stable non-zero safe blend weight or best_safe includes 028.",
        "should_replace_submission_slot": "Only if best safe blend beats 027 CV materially and Public LB confirms. 019 stays unless LB > 0.97000.",
    },
    "automatic_view": {
        "best_safe_includes_028": bool(best_info.get("includes_028")),
        "best_safe_weight_028": best_safe_weight_028,
        "best_028_safe_cv": None if best_028_safe is None else float(best_028_safe["balanced_accuracy"]),
        "best_028_safe_method": None if best_028_safe is None else best_028_safe["method"],
        "best_028_safe_set": None if best_028_safe is None else best_028_safe["set_name"],
    },
}
save_json(judgement, OUTDIR / "role_judgement.json")

summary = {
    "competition": COMPETITION,
    "exp_id": EXP_ID,
    "status": "completed",
    "purpose": "Blend/correlation check after adding 028 CatBoost v3 as-is to 019/027/026/024/020/017/015/018 candidate pool",
    "npy_dir": str(NPY_DIR),
    "model_keys": model_keys,
    "class_names": class_names,
    "resolved_specs": resolved_specs,
    "individual_scores": individual_df.to_dict(orient="records"),
    "pairwise_oof_correlation": pairwise_df.to_dict(orient="records"),
    "pairwise_oof_correlation_focus_028": focus_028_df.to_dict(orient="records"),
    "blend_diagnostics": blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics": safe_blend_df.to_dict(orient="records"),
    "safe_blend_diagnostics_focus_028": focus_028_blend_df.to_dict(orient="records"),
    "best_blend_info": best_info,
    "saved_submission_candidates": saved_submission_df.to_dict(orient="records"),
    "role_judgement": judgement,
}
save_json(summary, OUTDIR / "blend_add028_cat_v3_summary.json")

memo = {
    "experiment": {
        "id": EXP_ID,
        "title": "Blend check after adding 028 CatBoost v3",
        "competition": COMPETITION,
        "status": "completed",
        "metric": "balanced_accuracy_score",
        "created_at": "2026-06-08",
    },
    "objective": {
        "summary": (
            "Load OOF/pred probabilities from npy-files, add 028 CatBoost v3 as-is, "
            "and decide whether this CatBoost family model adds complementary signal to 019/027/026/024/020/017/015/018."
        ),
        "success_criteria": [
            "load 015/017/018/019/020/024/026/027/028 OOF and pred npy files",
            "recompute individual balanced accuracy",
            "compute pairwise OOF correlations and focus table for 028",
            "evaluate add028 blend sets",
            "separate safe blends from in-sample meta screening",
            "save best safe blend OOF/pred npy",
            "save best safe blend submission",
        ],
    },
    "inputs": {"npy_dir": str(NPY_DIR), "models": resolved_specs, "blend_sets": BLEND_SETS},
    "outputs": {
        "individual_scores": "individual_scores.csv",
        "pairwise_oof_correlation": "pairwise_oof_correlation.csv",
        "pairwise_oof_correlation_focus_028": "pairwise_oof_correlation_focus_028.csv",
        "blend_diagnostics": "blend_diagnostics.csv",
        "safe_blend_diagnostics": "safe_blend_diagnostics.csv",
        "safe_blend_diagnostics_focus_028": "safe_blend_diagnostics_focus_028.csv",
        "best_blend_info": "best_blend_info.json",
        "best_oof_proba": best_oof_npy.name,
        "best_pred_proba": best_pred_npy.name,
        "best_submission": best_submission_path.name,
        "summary": "blend_add028_cat_v3_summary.json",
    },
    "judgement": {
        "status": "completed_pending_manual_review",
        "initial_view": (
            "028 is a strong CatBoost v3 single model, but its standalone Public LB does not replace 019/027/026/017. "
            "Its value depends on whether it keeps nonzero weight in safe blends with 019/027 and current RealMLP/blend components."
        ),
        "automatic_view": judgement["automatic_view"],
        "next_action": [
            "Review individual_scores.csv",
            "Review pairwise_oof_correlation_focus_028.csv, especially 028 vs 019 and 027",
            "Review safe_blend_diagnostics_focus_028.csv",
            "Check whether best safe blend includes 028 and whether 028 weight is stable/non-trivial",
            "Submit best safe blend only if it improves over 027 CV enough and/or has credible Public LB upside",
            "Do not bias-search 028 unless it receives useful safe blend weight",
        ],
    },
}

memo_path = OUTDIR / "memo.yml"
if yaml is not None:
    with open(memo_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(memo, f, allow_unicode=True, sort_keys=False)
else:
    with open(memo_path, "w", encoding="utf-8") as f:
        f.write(json.dumps(memo, indent=2, ensure_ascii=False, default=str))

print("Saved files:")
for p in sorted(OUTDIR.glob("*")):
    print(" -", p.name)


Saved files:
 - best_blend_info.json
 - blend_add028_cat_v3_summary.json
 - blend_diagnostics.csv
 - individual_scores.csv
 - memo.yml
 - oof_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy
 - pairwise_oof_correlation.csv
 - pairwise_oof_correlation_focus_028.csv
 - pred_exp_20260608_029_blend_add028_cat_v3_check_best_blend_proba.npy
 - role_judgement.json
 - safe_blend_diagnostics.csv
 - safe_blend_diagnostics_focus_028.csv
 - saved_submission_candidates.csv
 - submission_exp_20260608_029_blend_add028_cat_v3_check_best_blend.csv
